# Enunciado — CRUD do minimundo do grupo

**Tarefa:** cada grupo deve criar um **CRUD em Python + PostgreSQL** para o banco do **minimundo do grupo** — o mesmo que vocês modelaram (DER → mapeamento → tabelas) ao longo da disciplina.

**Este notebook é um exemplo.** Ele usa o minimundo da **locadora de jogos** (`locadora-demo.sql`), que é o exemplo que resolvemos juntos em aula. O trabalho do grupo deve seguir a mesma estrutura, mas com as **tabelas, colunas e regras do minimundo de vocês**. Não é para copiar a locadora: é para usar como modelo.

## O que o notebook do grupo precisa ter

1. **Conexão** com o banco do minimundo do grupo.
2. **Para cada tabela**, nesta ordem:
   - **Classe** — uma classe Python representando uma linha da tabela (atributos = colunas);
   - **Métodos** — as quatro operações do CRUD: criar (`INSERT`), ler (`SELECT` — listar todos e buscar pela chave), atualizar (`UPDATE`) e apagar (`DELETE`);
   - **Como usar** — uma célula de exemplo que chama cada método e mostra o resultado.
3. **Menu** — no final de tudo, um programa interativo (`input`/`print`) que usa os métodos para as operações principais do minimundo de vocês.
4. **Documentação** em células Markdown explicando cada seção, e comentários nas linhas de código que não forem óbvias.

## Regras

- Os valores entram no SQL **sempre** com `%s` e uma tupla — nunca montando o comando com f-string (risco de *SQL Injection*).
- Todo método trata erro com `try`/`except pg.DatabaseError`, faz `commit()` quando dá certo e `rollback()` quando dá erro.
- As tabelas do meio (N:N) também têm CRUD. Se não tiverem atributo próprio, explique por que não têm Update.
- As restrições do banco (`PRIMARY KEY`, `FOREIGN KEY`, `UNIQUE`, `CHECK`, `NOT NULL`) devem aparecer funcionando em pelo menos um exemplo de "Como usar": mostre o banco **recusando** um dado inválido.

---

# Exemplo: CRUD da Locadora — Python + PostgreSQL

Notebook da disciplina de **Banco de Dados**. Aqui o banco da locadora (criado pelo arquivo `locadora-demo.sql`) ganha um **CRUD completo** em Python, usando a biblioteca `psycopg2`.

**CRUD** são as quatro operações básicas sobre os dados:

| Letra | Operação | Comando SQL |
|---|---|---|
| **C** | *Create* — criar | `INSERT` |
| **R** | *Read* — ler | `SELECT` |
| **U** | *Update* — atualizar | `UPDATE` |
| **D** | *Delete* — apagar | `DELETE` |

## Organização do notebook

- **1. Conexão** com o PostgreSQL
- **2. Padrões** usados em todas as classes e métodos
- **3 a 9. Uma seção por tabela**, sempre na mesma ordem:
  - **Classe** — a classe Python que representa uma linha da tabela
  - **Métodos** — as funções de CRUD da tabela
  - **Como usar** — um exemplo rodando cada método
- **10. Menu** — o programa interativo da locadora, que usa tudo o que veio antes

As tabelas aparecem na mesma ordem do `locadora-demo.sql`: primeiro quem é apontado, depois quem aponta. Assim, cada seção só usa métodos que já foram definidos acima dela.

## As sete tabelas

| Tabela | Chave primária | O que guarda |
|---|---|---|
| `plataforma` | `id_plataforma` | consoles / PC |
| `jogo` | `id_jogo` | os jogos do acervo |
| `cliente` | `id_cliente` | os clientes da locadora |
| `endereco` | `id_endereco` | endereço do cliente (**1:1** — `id_cliente` é `UNIQUE`) |
| `jogos_plataforma` | `(id_jogo, id_plataforma)` | **N:N** entre jogo e plataforma (tabela do meio, pura) |
| `emprestimo` | `id_emprestimo` | cada locação feita por um cliente |
| `item_emprestimo` | `(id_emprestimo, id_jogo)` | **N:N** entre empréstimo e jogo, com atributo próprio (`preco_pago`) |

In [ ]:
!pip install psycopg2

In [ ]:
import psycopg2 as pg
from datetime import date, timedelta
from decimal import Decimal  # o psycopg2 devolve colunas NUMERIC como Decimal

# 1. Conexão

`pg.connect(...)` abre a conexão com o servidor e `conexao.cursor()` cria o **cursor**, o objeto que executa os comandos SQL e traz os resultados.

> Se você criou o banco com o nome sugerido no `locadora-demo.sql`, troque `dbname="postgres"` por `dbname="locadora_demo"`.

In [ ]:
conexao = pg.connect(
    dbname="postgres",
    user="postgres",
    password="postgres",
    host="localhost",
    port="5432"
)
cursor = conexao.cursor()
print("conectado!!")

# 2. Padrões

## Classes

Cada classe representa **uma linha** da tabela de mesmo nome: os atributos são as colunas.

- O **id** é o último parâmetro do construtor e é opcional (`None`), porque quem gera o id é o banco (`SERIAL`). O objeto só recebe o id depois do `INSERT`.
- `__str__` mostra todos os campos (usado pelo `print`).
- `__repr__` mostra um resumo (usado quando o objeto aparece dentro de uma lista).

| Classe | Tabela |
|---|---|
| `Plataforma` | `plataforma` |
| `Jogo` | `jogo` |
| `Cliente` | `cliente` |
| `Endereco` | `endereco` |
| `JogoPlataforma` | `jogos_plataforma` |
| `Emprestimo` | `emprestimo` |
| `ItemEmprestimo` | `item_emprestimo` |

> **Mudança em relação à versão anterior:** a classe `Jogo` não tem mais `id_plataforma`. A tabela `jogo` não tem essa coluna — um jogo pode rodar em **várias** plataformas, então essa ligação fica na tabela `jogos_plataforma` (classe `JogoPlataforma`).

## Métodos (CRUD)

Todas as funções seguem o **mesmo padrão**:

```python
try:
    cursor.execute("SQL com %s", (valores,))   # 1. executa o comando
    ...                                        # 2. lê o resultado (fetchone / fetchall)
    conexao.commit()                           # 3. confirma a transação (só em INSERT/UPDATE/DELETE)
except pg.DatabaseError as e:
    print(e)                                   # mostra o erro que o banco devolveu
    conexao.rollback()                         # desfaz a transação
```

Pontos importantes:

- **Sempre `%s` e uma tupla de valores, nunca f-string** com os dados dentro do SQL. O `psycopg2` coloca os valores no lugar com as aspas e o escape certos, o que impede **SQL Injection**.
- **Tupla de um elemento leva vírgula:** `(id_jogo,)`. Sem a vírgula, `(id_jogo)` é só um número entre parênteses.
- **`RETURNING`** faz o `INSERT` devolver colunas geradas pelo banco (o id do `SERIAL`, os valores `DEFAULT`).
- **`cursor.rowcount`** diz quantas linhas o `UPDATE`/`DELETE` afetou. Se for `0`, o id não existe.
- **`rollback()` é obrigatório no erro:** depois de um erro, o PostgreSQL recusa qualquer comando até a transação ser desfeita.
- Nos `SELECT`, as colunas são escritas **por nome** (em vez de `SELECT *`), para a posição de cada uma em `linha[0]`, `linha[1]`... ser garantida.
- Quando um método faz **mais de um comando** (ex.: apagar os vínculos e depois o registro), o `commit()` só vem no final: ou tudo é gravado, ou nada é (**atomicidade**).

### Lista de métodos

| Tabela | Create | Read | Update | Delete |
|---|---|---|---|---|
| `plataforma` | `criar_plataforma` | `listar_plataformas`, `buscar_plataforma` | `atualizar_plataforma` | `deletar_plataforma` |
| `jogo` | `criar_jogo` | `listar_jogos`, `buscar_jogo` | `atualizar_jogo` | `deletar_jogo` |
| `cliente` | `criar_cliente` | `listar_clientes`, `buscar_cliente` | `atualizar_cliente`, `desativar_cliente` | `deletar_cliente` |
| `endereco` | `criar_endereco` | `listar_enderecos`, `buscar_endereco`, `buscar_endereco_do_cliente` | `atualizar_endereco` | `deletar_endereco` |
| `jogos_plataforma` | `vincular_jogo_plataforma` | `listar_jogos_plataforma`, `listar_plataformas_do_jogo`, `listar_jogos_da_plataforma` | — | `desvincular_jogo_plataforma` |
| `emprestimo` | `criar_emprestimo`, `registrar_aluguel` | `listar_emprestimos`, `buscar_emprestimo` | `atualizar_emprestimo`, `registrar_devolucao` | `deletar_emprestimo` |
| `item_emprestimo` | `criar_item_emprestimo` | `listar_itens_emprestimo` | `atualizar_item_emprestimo` | `deletar_item_emprestimo` |

## Como usar

Cada tabela termina com uma célula **Como usar**, com exemplos de cada método. Cada uma faz o ciclo completo — **cria, lê, atualiza e apaga** — com dados de teste, e no fim apaga o que criou. Assim dá para rodar quantas vezes quiser sem sujar o banco.

O fluxo de um Update é sempre o mesmo: **buscar → mudar o atributo no objeto → mandar atualizar.**

```python
jogo = buscar_jogo(1)
jogo.preco_diaria = Decimal("9.90")
atualizar_jogo(jogo)
```

> Valores em dinheiro usam `Decimal("12.50")`, e não `12.50`: o `float` não representa centavos com exatidão.

# 3. Plataforma

Os consoles e o PC. `nome` é `UNIQUE`: a mesma plataforma não entra duas vezes.

## Classe

In [ ]:
class Plataforma():
    id_plataforma: int | None
    nome: str
    fabricante: str | None

    def __init__(self, nome: str, fabricante: str | None, id_plataforma: int | None = None):
        self.id_plataforma = id_plataforma
        self.nome = nome
        self.fabricante = fabricante

    def __str__(self):
        return (f"Plataforma(id_plataforma={self.id_plataforma}, "
                f"nome={self.nome}, "
                f"fabricante={self.fabricante})")

    def __repr__(self):
        return f"Plataforma( {self.id_plataforma} - {self.nome} )"

## Métodos

| Método | SQL | Retorno |
|---|---|---|
| `criar_plataforma(nome, fabricante)` | `INSERT ... RETURNING id_plataforma` | `Plataforma` ou `None` |
| `listar_plataformas()` | `SELECT` de todas | `list[Plataforma]` |
| `buscar_plataforma(id_plataforma)` | `SELECT ... WHERE id_plataforma = %s` | `Plataforma` ou `None` |
| `atualizar_plataforma(plataforma)` | `UPDATE ... WHERE id_plataforma = %s` | `True` se alterou |
| `deletar_plataforma(id_plataforma)` | `DELETE` dos vínculos + `DELETE` da plataforma | `True` se apagou |

In [ ]:
def criar_plataforma(nome: str, fabricante: str | None) -> Plataforma | None:
    try:
        cursor.execute(
            "INSERT INTO plataforma (nome, fabricante) "
            "VALUES (%s, %s) RETURNING id_plataforma",
            (nome, fabricante)
        )
        id_plataforma = cursor.fetchone()[0]  # o id que o SERIAL acabou de gerar
        conexao.commit()
        return Plataforma(nome, fabricante, id_plataforma)
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return None


def listar_plataformas() -> list[Plataforma]:
    plataformas = []
    try:
        cursor.execute(
            "SELECT id_plataforma, nome, fabricante "
            "FROM plataforma ORDER BY id_plataforma"
        )
        for linha in cursor.fetchall():
            plataformas.append(Plataforma(linha[1], linha[2], linha[0]))
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return plataformas


def buscar_plataforma(id_plataforma: int) -> Plataforma | None:
    try:
        cursor.execute(
            "SELECT id_plataforma, nome, fabricante "
            "FROM plataforma WHERE id_plataforma = %s",
            (id_plataforma,)  # a vírgula faz disso uma tupla
        )
        linha = cursor.fetchone()  # None se nenhuma linha bateu com o WHERE
        if linha:
            return Plataforma(linha[1], linha[2], linha[0])
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return None


def atualizar_plataforma(plataforma: Plataforma) -> bool:
    try:
        cursor.execute(
            "UPDATE plataforma SET nome = %s, fabricante = %s "
            "WHERE id_plataforma = %s",
            (plataforma.nome, plataforma.fabricante, plataforma.id_plataforma)
        )
        alteradas = cursor.rowcount  # 0 = não existe plataforma com esse id
        conexao.commit()
        return alteradas > 0
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return False


def deletar_plataforma(id_plataforma: int) -> bool:
    try:
        # jogos_plataforma aponta para plataforma: os vínculos saem primeiro,
        # senão a chave estrangeira impede o DELETE
        cursor.execute("DELETE FROM jogos_plataforma WHERE id_plataforma = %s", (id_plataforma,))
        cursor.execute("DELETE FROM plataforma WHERE id_plataforma = %s", (id_plataforma,))
        apagadas = cursor.rowcount
        conexao.commit()  # confirma os dois DELETEs juntos
        return apagadas > 0
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return False

## Como usar

In [ ]:
print("todas:      ", listar_plataformas())                               # R

plataforma = criar_plataforma("Plataforma Teste", "Fabricante Teste")      # C
if plataforma:
    print("criada:     ", plataforma)

    plataforma.fabricante = "Outro Fabricante"
    print("atualizou?  ", atualizar_plataforma(plataforma))                # U
    print("buscada:    ", buscar_plataforma(plataforma.id_plataforma))     # R

    print("apagou?     ", deletar_plataforma(plataforma.id_plataforma))    # D
    print("depois:     ", buscar_plataforma(plataforma.id_plataforma))     # None: não existe mais

# 4. Jogo

Os jogos do acervo. `preco_diaria` é o preço de **hoje**; o que foi cobrado numa locação fica em `item_emprestimo`.

## Classe

In [ ]:
class Jogo():
    id_jogo: int | None
    titulo: str
    genero: str | None
    ano_lancamento: int | None
    preco_diaria: Decimal

    def __init__(
        self,
        titulo: str,
        genero: str | None,
        ano_lancamento: int | None,
        preco_diaria: Decimal,
        id_jogo: int | None = None
        ):
        self.id_jogo = id_jogo
        self.titulo = titulo
        self.genero = genero
        self.ano_lancamento = ano_lancamento
        self.preco_diaria = preco_diaria

    def __str__(self):
        return (f"Jogo(id_jogo={self.id_jogo}, "
                f"titulo={self.titulo}, "
                f"genero={self.genero}, "
                f"ano_lancamento={self.ano_lancamento}, "
                f"preco_diaria={self.preco_diaria})")

    def __repr__(self):
        return f"Jogo( {self.id_jogo} - {self.titulo} )"

## Métodos

| Método | SQL | Retorno |
|---|---|---|
| `criar_jogo(titulo, genero, ano_lancamento, preco_diaria)` | `INSERT ... RETURNING id_jogo` | `Jogo` ou `None` |
| `listar_jogos()` | `SELECT` de todos | `list[Jogo]` |
| `buscar_jogo(id_jogo)` | `SELECT ... WHERE id_jogo = %s` | `Jogo` ou `None` |
| `atualizar_jogo(jogo)` | `UPDATE ... WHERE id_jogo = %s` | `True` se alterou |
| `deletar_jogo(id_jogo)` | `DELETE` dos vínculos + `DELETE` do jogo | `True` se apagou |

> O banco tem `CHECK (ano_lancamento > 1970)` e `CHECK (preco_diaria >= 0)`. Se o valor violar a regra, o `INSERT`/`UPDATE` é recusado e o método devolve `None`/`False`.
>
> Um jogo que **já foi alugado** não pode ser apagado: `item_emprestimo` aponta para ele. O `DELETE` falha, o `rollback()` desfaz também a remoção dos vínculos, e o jogo fica como estava.

In [ ]:
def criar_jogo(
    titulo: str,
    genero: str | None,
    ano_lancamento: int | None,
    preco_diaria: Decimal
    ) -> Jogo | None:
    try:
        cursor.execute(
            "INSERT INTO jogo (titulo, genero, ano_lancamento, preco_diaria) "
            "VALUES (%s, %s, %s, %s) RETURNING id_jogo",
            (titulo, genero, ano_lancamento, preco_diaria)
        )
        id_jogo = cursor.fetchone()[0]
        conexao.commit()
        return Jogo(titulo, genero, ano_lancamento, preco_diaria, id_jogo)
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return None


def listar_jogos() -> list[Jogo]:
    jogos = []
    try:
        cursor.execute(
            "SELECT id_jogo, titulo, genero, ano_lancamento, preco_diaria "
            "FROM jogo ORDER BY id_jogo"
        )
        for linha in cursor.fetchall():
            jogos.append(Jogo(linha[1], linha[2], linha[3], linha[4], linha[0]))
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return jogos


def buscar_jogo(id_jogo: int) -> Jogo | None:
    try:
        cursor.execute(
            "SELECT id_jogo, titulo, genero, ano_lancamento, preco_diaria "
            "FROM jogo WHERE id_jogo = %s",
            (id_jogo,)
        )
        linha = cursor.fetchone()
        if linha:
            return Jogo(linha[1], linha[2], linha[3], linha[4], linha[0])
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return None


def atualizar_jogo(jogo: Jogo) -> bool:
    try:
        cursor.execute(
            "UPDATE jogo SET titulo = %s, genero = %s, ano_lancamento = %s, preco_diaria = %s "
            "WHERE id_jogo = %s",
            (jogo.titulo, jogo.genero, jogo.ano_lancamento, jogo.preco_diaria, jogo.id_jogo)
        )
        alteradas = cursor.rowcount
        conexao.commit()
        return alteradas > 0
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return False


def deletar_jogo(id_jogo: int) -> bool:
    try:
        cursor.execute("DELETE FROM jogos_plataforma WHERE id_jogo = %s", (id_jogo,))
        cursor.execute("DELETE FROM jogo WHERE id_jogo = %s", (id_jogo,))
        apagadas = cursor.rowcount
        conexao.commit()
        return apagadas > 0
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()  # desfaz também o DELETE dos vínculos
    return False

## Como usar

A segunda tentativa usa `ano_lancamento = 1950` de propósito: o `CHECK (ano_lancamento > 1970)` recusa a linha e o método devolve `None`.

In [ ]:
print("todos:      ", listar_jogos())

jogo = criar_jogo("Jogo Teste", "Aventura", 2024, Decimal("10.00"))
if jogo:
    print("criado:     ", jogo)

    jogo.preco_diaria = Decimal("12.50")
    print("atualizou?  ", atualizar_jogo(jogo))
    print("buscado:    ", buscar_jogo(jogo.id_jogo))

    print("apagou?     ", deletar_jogo(jogo.id_jogo))

print("ano inválido:", criar_jogo("Jogo Antigo", "Arcade", 1950, Decimal("5.00")))

# 5. Cliente

Os clientes da locadora. `(nome, telefone)` é `UNIQUE`: a mesma pessoa não é cadastrada duas vezes.

## Classe

`data_cadastro` e `ativo` têm valor padrão **no banco** (`DEFAULT CURRENT_DATE` e `DEFAULT TRUE`), por isso são opcionais aqui.

In [ ]:
class Cliente():
    id_cliente: int | None
    nome: str
    telefone: str | None
    email: str | None
    data_cadastro: date | None
    ativo: bool

    def __init__(
        self,
        nome: str,
        telefone: str | None = None,
        email: str | None = None,
        data_cadastro: date | None = None,
        ativo: bool = True,
        id_cliente: int | None = None
        ):
        self.id_cliente = id_cliente
        self.nome = nome
        self.telefone = telefone
        self.email = email
        self.data_cadastro = data_cadastro
        self.ativo = ativo

    def __str__(self):
        return (f"Cliente(id_cliente={self.id_cliente}, "
                f"nome={self.nome}, "
                f"telefone={self.telefone}, "
                f"email={self.email}, "
                f"data_cadastro={self.data_cadastro}, "
                f"ativo={self.ativo})")

    def __repr__(self):
        return f"Cliente( {self.id_cliente} - {self.nome} )"

## Métodos

| Método | SQL | Retorno |
|---|---|---|
| `criar_cliente(nome, telefone, email)` | `INSERT ... RETURNING id_cliente, data_cadastro, ativo` | `Cliente` ou `None` |
| `listar_clientes(apenas_ativos)` | `SELECT` de todos (ou só `WHERE ativo`) | `list[Cliente]` |
| `buscar_cliente(id_cliente)` | `SELECT ... WHERE id_cliente = %s` | `Cliente` ou `None` |
| `atualizar_cliente(cliente)` | `UPDATE ... WHERE id_cliente = %s` | `True` se alterou |
| `desativar_cliente(id_cliente)` | `UPDATE cliente SET ativo = FALSE` | `True` se alterou |
| `deletar_cliente(id_cliente)` | `DELETE` do endereço + `DELETE` do cliente | `True` se apagou |

**Apagar × desativar.** Um cliente com empréstimos **não pode ser apagado** (a chave estrangeira em `emprestimo` impede), e nem deveria: o histórico de locações sumiria. Por isso a tabela tem a coluna `ativo`. "Remover" um cliente na prática é `desativar_cliente`, a chamada **exclusão lógica** (*soft delete*).

In [ ]:
def criar_cliente(nome: str, telefone: str | None = None, email: str | None = None) -> Cliente | None:
    try:
        # data_cadastro e ativo ficam de fora do INSERT para o DEFAULT do banco agir;
        # o RETURNING traz os valores que ele preencheu
        cursor.execute(
            "INSERT INTO cliente (nome, telefone, email) "
            "VALUES (%s, %s, %s) RETURNING id_cliente, data_cadastro, ativo",
            (nome, telefone, email)
        )
        id_cliente, data_cadastro, ativo = cursor.fetchone()
        conexao.commit()
        return Cliente(nome, telefone, email, data_cadastro, ativo, id_cliente)
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return None


def listar_clientes(apenas_ativos: bool = False) -> list[Cliente]:
    clientes = []
    sql = "SELECT id_cliente, nome, telefone, email, data_cadastro, ativo FROM cliente"
    if apenas_ativos:
        sql += " WHERE ativo"  # texto fixo, sem dado do usuário: concatenar aqui é seguro
    sql += " ORDER BY id_cliente"
    try:
        cursor.execute(sql)
        for linha in cursor.fetchall():
            clientes.append(Cliente(linha[1], linha[2], linha[3], linha[4], linha[5], linha[0]))
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return clientes


def buscar_cliente(id_cliente: int) -> Cliente | None:
    try:
        cursor.execute(
            "SELECT id_cliente, nome, telefone, email, data_cadastro, ativo "
            "FROM cliente WHERE id_cliente = %s",
            (id_cliente,)
        )
        linha = cursor.fetchone()
        if linha:
            return Cliente(linha[1], linha[2], linha[3], linha[4], linha[5], linha[0])
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return None


def atualizar_cliente(cliente: Cliente) -> bool:
    try:
        cursor.execute(
            "UPDATE cliente SET nome = %s, telefone = %s, email = %s, ativo = %s "
            "WHERE id_cliente = %s",
            (cliente.nome, cliente.telefone, cliente.email, cliente.ativo, cliente.id_cliente)
        )
        alteradas = cursor.rowcount
        conexao.commit()
        return alteradas > 0
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return False


def desativar_cliente(id_cliente: int) -> bool:
    try:
        cursor.execute("UPDATE cliente SET ativo = FALSE WHERE id_cliente = %s", (id_cliente,))
        alteradas = cursor.rowcount
        conexao.commit()
        return alteradas > 0
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return False


def deletar_cliente(id_cliente: int) -> bool:
    try:
        # o endereço (1:1) aponta para o cliente, então sai primeiro.
        # Se o cliente tiver empréstimos, o segundo DELETE falha e o rollback
        # devolve o endereço também: nada é apagado pela metade.
        cursor.execute("DELETE FROM endereco WHERE id_cliente = %s", (id_cliente,))
        cursor.execute("DELETE FROM cliente WHERE id_cliente = %s", (id_cliente,))
        apagadas = cursor.rowcount
        conexao.commit()
        return apagadas > 0
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return False

## Como usar

In [ ]:
print("todos:      ", listar_clientes())

cliente = criar_cliente("Cliente Teste", "(51) 90000-0000", "teste@exemplo.com")
if cliente:
    print("criado:     ", cliente)  # data_cadastro e ativo vieram do DEFAULT do banco

    cliente.telefone = "(51) 91111-1111"
    print("atualizou?  ", atualizar_cliente(cliente))

    print("desativou?  ", desativar_cliente(cliente.id_cliente))
    print("buscado:    ", buscar_cliente(cliente.id_cliente))  # agora com ativo=False

    print("apagou?     ", deletar_cliente(cliente.id_cliente))

# 6. Endereco

O endereço do cliente. Relacionamento **1:1** com `cliente` (opcional: nem todo cliente tem endereço).

## Classe

In [ ]:
class Endereco():
    id_endereco: int | None
    id_cliente: int
    logradouro: str
    numero: str | None
    bairro: str | None
    cidade: str
    uf: str | None
    cep: str | None

    def __init__(
        self,
        id_cliente: int,
        logradouro: str,
        cidade: str,
        numero: str | None = None,
        bairro: str | None = None,
        uf: str | None = None,
        cep: str | None = None,
        id_endereco: int | None = None
        ):
        self.id_endereco = id_endereco
        self.id_cliente = id_cliente
        self.logradouro = logradouro
        self.numero = numero
        self.bairro = bairro
        self.cidade = cidade
        self.uf = uf
        self.cep = cep

    def __str__(self):
        return (f"Endereco(id_endereco={self.id_endereco}, "
                f"id_cliente={self.id_cliente}, "
                f"logradouro={self.logradouro}, "
                f"numero={self.numero}, "
                f"bairro={self.bairro}, "
                f"cidade={self.cidade}, "
                f"uf={self.uf}, "
                f"cep={self.cep})")

    def __repr__(self):
        return f"Endereco( {self.id_endereco} - {self.logradouro}, {self.numero} - {self.cidade} )"

## Métodos

| Método | SQL | Retorno |
|---|---|---|
| `criar_endereco(id_cliente, logradouro, cidade, ...)` | `INSERT ... RETURNING id_endereco` | `Endereco` ou `None` |
| `listar_enderecos()` | `SELECT` de todos | `list[Endereco]` |
| `buscar_endereco(id_endereco)` | `SELECT ... WHERE id_endereco = %s` | `Endereco` ou `None` |
| `buscar_endereco_do_cliente(id_cliente)` | `SELECT ... WHERE id_cliente = %s` | `Endereco` ou `None` |
| `atualizar_endereco(endereco)` | `UPDATE ... WHERE id_endereco = %s` | `True` se alterou |
| `deletar_endereco(id_endereco)` | `DELETE ... WHERE id_endereco = %s` | `True` se apagou |

> O relacionamento é **1:1** por causa do `UNIQUE` em `endereco.id_cliente`: tentar cadastrar um segundo endereço para o mesmo cliente dá erro de chave duplicada. Por isso `buscar_endereco_do_cliente` usa `fetchone()` — no máximo uma linha volta.

In [ ]:
def _linha_para_endereco(linha) -> Endereco:
    # ordem das colunas do SELECT: id_endereco, id_cliente, logradouro, numero, bairro, cidade, uf, cep
    return Endereco(
        id_cliente=linha[1],
        logradouro=linha[2],
        cidade=linha[5],
        numero=linha[3],
        bairro=linha[4],
        uf=linha[6],
        cep=linha[7],
        id_endereco=linha[0]
    )


SELECT_ENDERECO = (
    "SELECT id_endereco, id_cliente, logradouro, numero, bairro, cidade, uf, cep "
    "FROM endereco"
)


def criar_endereco(
    id_cliente: int,
    logradouro: str,
    cidade: str,
    numero: str | None = None,
    bairro: str | None = None,
    uf: str | None = None,
    cep: str | None = None
    ) -> Endereco | None:
    try:
        cursor.execute(
            "INSERT INTO endereco (id_cliente, logradouro, numero, bairro, cidade, uf, cep) "
            "VALUES (%s, %s, %s, %s, %s, %s, %s) RETURNING id_endereco",
            (id_cliente, logradouro, numero, bairro, cidade, uf, cep)
        )
        id_endereco = cursor.fetchone()[0]
        conexao.commit()
        return Endereco(id_cliente, logradouro, cidade, numero, bairro, uf, cep, id_endereco)
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return None


def listar_enderecos() -> list[Endereco]:
    enderecos = []
    try:
        cursor.execute(SELECT_ENDERECO + " ORDER BY id_endereco")
        for linha in cursor.fetchall():
            enderecos.append(_linha_para_endereco(linha))
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return enderecos


def buscar_endereco(id_endereco: int) -> Endereco | None:
    try:
        cursor.execute(SELECT_ENDERECO + " WHERE id_endereco = %s", (id_endereco,))
        linha = cursor.fetchone()
        if linha:
            return _linha_para_endereco(linha)
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return None


def buscar_endereco_do_cliente(id_cliente: int) -> Endereco | None:
    try:
        cursor.execute(SELECT_ENDERECO + " WHERE id_cliente = %s", (id_cliente,))
        linha = cursor.fetchone()
        if linha:
            return _linha_para_endereco(linha)
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return None


def atualizar_endereco(endereco: Endereco) -> bool:
    try:
        cursor.execute(
            "UPDATE endereco SET logradouro = %s, numero = %s, bairro = %s, "
            "cidade = %s, uf = %s, cep = %s "
            "WHERE id_endereco = %s",
            (endereco.logradouro, endereco.numero, endereco.bairro,
             endereco.cidade, endereco.uf, endereco.cep, endereco.id_endereco)
        )
        alteradas = cursor.rowcount
        conexao.commit()
        return alteradas > 0
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return False


def deletar_endereco(id_endereco: int) -> bool:
    try:
        cursor.execute("DELETE FROM endereco WHERE id_endereco = %s", (id_endereco,))
        apagadas = cursor.rowcount
        conexao.commit()
        return apagadas > 0
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return False

## Como usar

O segundo `criar_endereco` para o **mesmo** cliente é recusado pelo `UNIQUE` — é isso que faz o relacionamento ser 1:1.

In [ ]:
print("todos:       ", listar_enderecos())

cliente = criar_cliente("Cliente Endereco Teste", "(51) 92222-2222")
if cliente:
    endereco = criar_endereco(cliente.id_cliente, "Rua Teste", "Porto Alegre", "100", "Centro", "RS", "90000-000")
    print("criado:      ", endereco)

    print("segundo end.:", criar_endereco(cliente.id_cliente, "Outra Rua", "Canoas"))  # None (UNIQUE)

    endereco.numero = "200"
    print("atualizou?   ", atualizar_endereco(endereco))
    print("do cliente:  ", buscar_endereco_do_cliente(cliente.id_cliente))

    print("apagou?      ", deletar_endereco(endereco.id_endereco))
    deletar_cliente(cliente.id_cliente)  # limpeza

# 7. JogosPlataforma

Tabela do meio do **N:N** entre `jogo` e `plataforma`: um jogo roda em várias plataformas, uma plataforma roda vários jogos.

## Classe

Tabela do meio **pura**: só as duas chaves estrangeiras, que juntas formam a chave primária. Por isso não tem id próprio.

In [ ]:
class JogoPlataforma():
    id_jogo: int
    id_plataforma: int

    def __init__(self, id_jogo: int, id_plataforma: int):
        self.id_jogo = id_jogo
        self.id_plataforma = id_plataforma

    def __str__(self):
        return f"JogoPlataforma(id_jogo={self.id_jogo}, id_plataforma={self.id_plataforma})"

    def __repr__(self):
        return f"JogoPlataforma( {self.id_jogo} - {self.id_plataforma} )"

## Métodos

| Método | SQL | Retorno |
|---|---|---|
| `vincular_jogo_plataforma(id_jogo, id_plataforma)` | `INSERT` | `JogoPlataforma` ou `None` |
| `listar_jogos_plataforma()` | `SELECT` de todos os vínculos | `list[JogoPlataforma]` |
| `listar_plataformas_do_jogo(id_jogo)` | `SELECT ... JOIN plataforma` | `list[Plataforma]` |
| `listar_jogos_da_plataforma(id_plataforma)` | `SELECT ... JOIN jogo` | `list[Jogo]` |
| `desvincular_jogo_plataforma(id_jogo, id_plataforma)` | `DELETE` | `True` se apagou |

**Por que não tem Update?** A tabela só tem as duas colunas da chave primária, sem nenhum atributo próprio. "Mudar" um vínculo é trocar a chave — o que, na prática, é **apagar um vínculo e criar outro**.

**JOIN:** `listar_plataformas_do_jogo` junta `jogos_plataforma` com `plataforma` para devolver as plataformas completas, e não só os ids.

In [ ]:
def vincular_jogo_plataforma(id_jogo: int, id_plataforma: int) -> JogoPlataforma | None:
    try:
        cursor.execute(
            "INSERT INTO jogos_plataforma (id_jogo, id_plataforma) VALUES (%s, %s)",
            (id_jogo, id_plataforma)
        )
        conexao.commit()
        return JogoPlataforma(id_jogo, id_plataforma)
    except pg.DatabaseError as e:
        print(e)  # ex.: vínculo repetido (PK) ou id inexistente (FK)
        conexao.rollback()
    return None


def listar_jogos_plataforma() -> list[JogoPlataforma]:
    vinculos = []
    try:
        cursor.execute(
            "SELECT id_jogo, id_plataforma FROM jogos_plataforma "
            "ORDER BY id_jogo, id_plataforma"
        )
        for linha in cursor.fetchall():
            vinculos.append(JogoPlataforma(linha[0], linha[1]))
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return vinculos


def listar_plataformas_do_jogo(id_jogo: int) -> list[Plataforma]:
    plataformas = []
    try:
        cursor.execute(
            "SELECT p.id_plataforma, p.nome, p.fabricante "
            "FROM jogos_plataforma jp "
            "JOIN plataforma p ON p.id_plataforma = jp.id_plataforma "
            "WHERE jp.id_jogo = %s "
            "ORDER BY p.nome",
            (id_jogo,)
        )
        for linha in cursor.fetchall():
            plataformas.append(Plataforma(linha[1], linha[2], linha[0]))
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return plataformas


def listar_jogos_da_plataforma(id_plataforma: int) -> list[Jogo]:
    jogos = []
    try:
        cursor.execute(
            "SELECT j.id_jogo, j.titulo, j.genero, j.ano_lancamento, j.preco_diaria "
            "FROM jogos_plataforma jp "
            "JOIN jogo j ON j.id_jogo = jp.id_jogo "
            "WHERE jp.id_plataforma = %s "
            "ORDER BY j.titulo",
            (id_plataforma,)
        )
        for linha in cursor.fetchall():
            jogos.append(Jogo(linha[1], linha[2], linha[3], linha[4], linha[0]))
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return jogos


def desvincular_jogo_plataforma(id_jogo: int, id_plataforma: int) -> bool:
    try:
        cursor.execute(
            "DELETE FROM jogos_plataforma WHERE id_jogo = %s AND id_plataforma = %s",
            (id_jogo, id_plataforma)  # chave composta: precisa das duas colunas no WHERE
        )
        apagadas = cursor.rowcount
        conexao.commit()
        return apagadas > 0
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return False

## Como usar

In [ ]:
print("todos os vínculos:", listar_jogos_plataforma())

jogo = criar_jogo("Jogo Multiplataforma Teste", "Corrida", 2023, Decimal("8.00"))
plat_a = criar_plataforma("Plataforma A Teste", "Fabricante A")
plat_b = criar_plataforma("Plataforma B Teste", "Fabricante B")

if jogo and plat_a and plat_b:
    print(vincular_jogo_plataforma(jogo.id_jogo, plat_a.id_plataforma))
    print(vincular_jogo_plataforma(jogo.id_jogo, plat_b.id_plataforma))
    print("repetido:     ", vincular_jogo_plataforma(jogo.id_jogo, plat_a.id_plataforma))  # None (PK)

    print("plataformas do jogo:", listar_plataformas_do_jogo(jogo.id_jogo))
    print("jogos da plat. A:   ", listar_jogos_da_plataforma(plat_a.id_plataforma))

    print("desvinculou?  ", desvincular_jogo_plataforma(jogo.id_jogo, plat_a.id_plataforma))
    print("plataformas do jogo:", listar_plataformas_do_jogo(jogo.id_jogo))

# limpeza: deletar_jogo já remove os vínculos que sobraram
if jogo:
    deletar_jogo(jogo.id_jogo)
if plat_a:
    deletar_plataforma(plat_a.id_plataforma)
if plat_b:
    deletar_plataforma(plat_b.id_plataforma)

# 8. Emprestimo

Cada locação feita por um cliente: quando retirou, quando deve devolver, quando devolveu e a multa.

## Classe

`data_entregue = None` quer dizer **"ainda não devolveu"** (no banco, `NULL`).

In [ ]:
class Emprestimo():
    id_emprestimo: int | None
    id_cliente: int
    data_retirada: date
    data_prevista: date
    data_entregue: date | None
    multa: Decimal

    def __init__(
        self,
        id_cliente: int,
        data_retirada: date,
        data_prevista: date,
        data_entregue: date | None = None,
        multa: Decimal = Decimal("0"),
        id_emprestimo: int | None = None
        ):
        self.id_emprestimo = id_emprestimo
        self.id_cliente = id_cliente
        self.data_retirada = data_retirada
        self.data_prevista = data_prevista
        self.data_entregue = data_entregue
        self.multa = multa

    def __str__(self):
        return (f"Emprestimo(id_emprestimo={self.id_emprestimo}, "
                f"id_cliente={self.id_cliente}, "
                f"data_retirada={self.data_retirada}, "
                f"data_prevista={self.data_prevista}, "
                f"data_entregue={self.data_entregue}, "
                f"multa={self.multa})")

    def __repr__(self):
        situacao = "aberto" if self.data_entregue is None else "devolvido"
        return f"Emprestimo( {self.id_emprestimo} - cliente {self.id_cliente} - {situacao} )"

## Métodos

| Método | SQL | Retorno |
|---|---|---|
| `criar_emprestimo(id_cliente, data_retirada, data_prevista)` | `INSERT ... RETURNING id_emprestimo` | `Emprestimo` ou `None` |
| `registrar_aluguel(id_cliente, ids_jogos, dias)` | `INSERT` do empréstimo + `INSERT` de cada item, **numa transação só** | `Emprestimo` ou `None` |
| `listar_emprestimos(apenas_abertos)` | `SELECT` de todos (ou só `WHERE data_entregue IS NULL`) | `list[Emprestimo]` |
| `buscar_emprestimo(id_emprestimo)` | `SELECT ... WHERE id_emprestimo = %s` | `Emprestimo` ou `None` |
| `atualizar_emprestimo(emprestimo)` | `UPDATE ... WHERE id_emprestimo = %s` | `True` se alterou |
| `registrar_devolucao(id_emprestimo, data_entregue, multa)` | `UPDATE` de `data_entregue` e `multa` | `True` se alterou |
| `deletar_emprestimo(id_emprestimo)` | `DELETE` dos itens + `DELETE` do empréstimo | `True` se apagou |

Regras que o **banco** garante (e que aparecem como erro no `print(e)`):

- `CHECK (data_prevista >= data_retirada)` — a devolução prevista não pode ser antes da retirada.
- `UNIQUE (id_cliente, data_retirada)` — o mesmo cliente não faz dois empréstimos no mesmo dia.
- `CHECK (multa >= 0)`.

**`registrar_aluguel` e as transações:** um aluguel é o empréstimo **mais** os seus itens. Se um dos jogos não existir, não faz sentido sobrar um empréstimo vazio no banco. Por isso todos os `INSERT` vão na mesma transação e o `commit()` só acontece no fim: se qualquer passo falha, o `rollback()` desfaz tudo.

In [ ]:
SELECT_EMPRESTIMO = (
    "SELECT id_emprestimo, id_cliente, data_retirada, data_prevista, data_entregue, multa "
    "FROM emprestimo"
)


def _linha_para_emprestimo(linha) -> Emprestimo:
    return Emprestimo(linha[1], linha[2], linha[3], linha[4], linha[5], linha[0])


def criar_emprestimo(id_cliente: int, data_retirada: date, data_prevista: date) -> Emprestimo | None:
    try:
        cursor.execute(
            "INSERT INTO emprestimo (id_cliente, data_retirada, data_prevista) "
            "VALUES (%s, %s, %s) RETURNING id_emprestimo, multa",
            (id_cliente, data_retirada, data_prevista)  # o psycopg2 converte date para DATE
        )
        id_emprestimo, multa = cursor.fetchone()
        conexao.commit()
        return Emprestimo(id_cliente, data_retirada, data_prevista, None, multa, id_emprestimo)
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return None


def registrar_aluguel(id_cliente: int, ids_jogos: list[int], dias: int = 3) -> Emprestimo | None:
    hoje = date.today()
    prevista = hoje + timedelta(days=dias)
    try:
        cursor.execute(
            "INSERT INTO emprestimo (id_cliente, data_retirada, data_prevista) "
            "VALUES (%s, %s, %s) RETURNING id_emprestimo, multa",
            (id_cliente, hoje, prevista)
        )
        id_emprestimo, multa = cursor.fetchone()

        for id_jogo in ids_jogos:
            # INSERT ... SELECT: copia o preco_diaria ATUAL do jogo para preco_pago (preço congelado)
            cursor.execute(
                "INSERT INTO item_emprestimo (id_emprestimo, id_jogo, preco_pago) "
                "SELECT %s, id_jogo, preco_diaria FROM jogo WHERE id_jogo = %s",
                (id_emprestimo, id_jogo)
            )
            if cursor.rowcount == 0:  # o SELECT não achou o jogo, então nada foi inserido
                raise pg.DatabaseError(f"Jogo {id_jogo} não existe")

        conexao.commit()  # só aqui o empréstimo e todos os itens passam a existir
        return Emprestimo(id_cliente, hoje, prevista, None, multa, id_emprestimo)
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()  # desfaz o empréstimo e os itens que já tinham entrado
    return None


def listar_emprestimos(apenas_abertos: bool = False) -> list[Emprestimo]:
    emprestimos = []
    sql = SELECT_EMPRESTIMO
    if apenas_abertos:
        sql += " WHERE data_entregue IS NULL"  # NULL se testa com IS NULL, nunca com = NULL
    sql += " ORDER BY id_emprestimo"
    try:
        cursor.execute(sql)
        for linha in cursor.fetchall():
            emprestimos.append(_linha_para_emprestimo(linha))
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return emprestimos


def buscar_emprestimo(id_emprestimo: int) -> Emprestimo | None:
    try:
        cursor.execute(SELECT_EMPRESTIMO + " WHERE id_emprestimo = %s", (id_emprestimo,))
        linha = cursor.fetchone()
        if linha:
            return _linha_para_emprestimo(linha)
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return None


def atualizar_emprestimo(emprestimo: Emprestimo) -> bool:
    try:
        cursor.execute(
            "UPDATE emprestimo SET id_cliente = %s, data_retirada = %s, data_prevista = %s, "
            "data_entregue = %s, multa = %s "
            "WHERE id_emprestimo = %s",
            (emprestimo.id_cliente, emprestimo.data_retirada, emprestimo.data_prevista,
             emprestimo.data_entregue, emprestimo.multa, emprestimo.id_emprestimo)
        )
        alteradas = cursor.rowcount
        conexao.commit()
        return alteradas > 0
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return False


def registrar_devolucao(
    id_emprestimo: int,
    data_entregue: date | None = None,
    multa: Decimal = Decimal("0")
    ) -> bool:
    if data_entregue is None:
        data_entregue = date.today()
    try:
        cursor.execute(
            "UPDATE emprestimo SET data_entregue = %s, multa = %s "
            "WHERE id_emprestimo = %s AND data_entregue IS NULL",  # só devolve o que está em aberto
            (data_entregue, multa, id_emprestimo)
        )
        alteradas = cursor.rowcount
        conexao.commit()
        return alteradas > 0
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return False


def deletar_emprestimo(id_emprestimo: int) -> bool:
    try:
        cursor.execute("DELETE FROM item_emprestimo WHERE id_emprestimo = %s", (id_emprestimo,))
        cursor.execute("DELETE FROM emprestimo WHERE id_emprestimo = %s", (id_emprestimo,))
        apagadas = cursor.rowcount
        conexao.commit()
        return apagadas > 0
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return False

## Como usar

Aqui aparece a **integridade referencial** funcionando: enquanto o cliente tem um empréstimo, `deletar_cliente` falha e devolve `False` — e o cliente continua lá. Só depois de apagar o empréstimo o cliente pode ser apagado.

In [ ]:
print("em aberto:       ", listar_emprestimos(apenas_abertos=True))

cliente = criar_cliente("Cliente Emprestimo Teste", "(51) 93333-3333")
if cliente:
    emprestimo = criar_emprestimo(cliente.id_cliente, date(2026, 10, 1), date(2026, 10, 4))
    print("criado:          ", emprestimo)

    # data_prevista antes da data_retirada: o CHECK recusa
    print("datas invertidas:", criar_emprestimo(cliente.id_cliente, date(2026, 10, 10), date(2026, 10, 5)))

    print("devolveu?        ", registrar_devolucao(emprestimo.id_emprestimo, date(2026, 10, 6), Decimal("6.00")))
    print("buscado:         ", buscar_emprestimo(emprestimo.id_emprestimo))

    print("apagou cliente?  ", deletar_cliente(cliente.id_cliente))  # False: tem empréstimo
    print("cliente ainda lá:", buscar_cliente(cliente.id_cliente))

    print("apagou empr.?    ", deletar_emprestimo(emprestimo.id_emprestimo))
    print("apagou cliente?  ", deletar_cliente(cliente.id_cliente))  # agora True

# 9. ItemEmprestimo

Tabela do meio do **N:N** entre `emprestimo` e `jogo`: quais jogos saíram em cada empréstimo, e por quanto.

## Classe

Tabela do meio **com atributo próprio**: `preco_pago` é o preço *congelado* no momento da locação (pode ser diferente do `preco_diaria` atual do jogo).

In [ ]:
class ItemEmprestimo():
    id_emprestimo: int
    id_jogo: int
    preco_pago: Decimal

    def __init__(self, id_emprestimo: int, id_jogo: int, preco_pago: Decimal):
        self.id_emprestimo = id_emprestimo
        self.id_jogo = id_jogo
        self.preco_pago = preco_pago

    def __str__(self):
        return (f"ItemEmprestimo(id_emprestimo={self.id_emprestimo}, "
                f"id_jogo={self.id_jogo}, "
                f"preco_pago={self.preco_pago})")

    def __repr__(self):
        return f"ItemEmprestimo( {self.id_emprestimo} - {self.id_jogo} - R$ {self.preco_pago} )"

## Métodos

| Método | SQL | Retorno |
|---|---|---|
| `criar_item_emprestimo(id_emprestimo, id_jogo, preco_pago)` | `INSERT ... SELECT` | `ItemEmprestimo` ou `None` |
| `listar_itens_emprestimo(id_emprestimo)` | `SELECT` de todos (ou só de um empréstimo) | `list[ItemEmprestimo]` |
| `atualizar_item_emprestimo(item)` | `UPDATE preco_pago ... WHERE id_emprestimo = %s AND id_jogo = %s` | `True` se alterou |
| `deletar_item_emprestimo(id_emprestimo, id_jogo)` | `DELETE` | `True` se apagou |

Se `preco_pago` não for informado, o método usa o `preco_diaria` atual do jogo, com `COALESCE(%s, preco_diaria)`: o `COALESCE` devolve o primeiro valor que não é `NULL`.

In [ ]:
def criar_item_emprestimo(
    id_emprestimo: int,
    id_jogo: int,
    preco_pago: Decimal | None = None
    ) -> ItemEmprestimo | None:
    try:
        cursor.execute(
            "INSERT INTO item_emprestimo (id_emprestimo, id_jogo, preco_pago) "
            "SELECT %s, id_jogo, COALESCE(%s, preco_diaria) FROM jogo WHERE id_jogo = %s "
            "RETURNING preco_pago",
            (id_emprestimo, preco_pago, id_jogo)
        )
        linha = cursor.fetchone()
        if linha is None:  # o jogo não existe: o SELECT não trouxe nada para inserir
            print(f"Jogo {id_jogo} não existe")
            conexao.rollback()
            return None
        conexao.commit()
        return ItemEmprestimo(id_emprestimo, id_jogo, linha[0])
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return None


def listar_itens_emprestimo(id_emprestimo: int | None = None) -> list[ItemEmprestimo]:
    itens = []
    try:
        if id_emprestimo is None:
            cursor.execute(
                "SELECT id_emprestimo, id_jogo, preco_pago FROM item_emprestimo "
                "ORDER BY id_emprestimo, id_jogo"
            )
        else:
            cursor.execute(
                "SELECT id_emprestimo, id_jogo, preco_pago FROM item_emprestimo "
                "WHERE id_emprestimo = %s ORDER BY id_jogo",
                (id_emprestimo,)
            )
        for linha in cursor.fetchall():
            itens.append(ItemEmprestimo(linha[0], linha[1], linha[2]))
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return itens


def atualizar_item_emprestimo(item: ItemEmprestimo) -> bool:
    try:
        cursor.execute(
            "UPDATE item_emprestimo SET preco_pago = %s "
            "WHERE id_emprestimo = %s AND id_jogo = %s",
            (item.preco_pago, item.id_emprestimo, item.id_jogo)
        )
        alteradas = cursor.rowcount
        conexao.commit()
        return alteradas > 0
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return False


def deletar_item_emprestimo(id_emprestimo: int, id_jogo: int) -> bool:
    try:
        cursor.execute(
            "DELETE FROM item_emprestimo WHERE id_emprestimo = %s AND id_jogo = %s",
            (id_emprestimo, id_jogo)
        )
        apagadas = cursor.rowcount
        conexao.commit()
        return apagadas > 0
    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
    return False

## Como usar

In [ ]:
print("todos os itens:  ", listar_itens_emprestimo())

cliente = criar_cliente("Cliente Item Teste", "(51) 94444-4444")
jogo_1 = criar_jogo("Jogo Item Teste 1", "RPG", 2022, Decimal("15.00"))
jogo_2 = criar_jogo("Jogo Item Teste 2", "Puzzle", 2021, Decimal("7.00"))

if cliente and jogo_1 and jogo_2:
    # aluguel completo: empréstimo + itens numa transação só
    emprestimo = registrar_aluguel(cliente.id_cliente, [jogo_1.id_jogo], dias=5)
    print("aluguel:        ", emprestimo)

    # mais um item, agora com preço promocional
    print("item promoção:  ", criar_item_emprestimo(emprestimo.id_emprestimo, jogo_2.id_jogo, Decimal("5.00")))
    print("itens:          ", listar_itens_emprestimo(emprestimo.id_emprestimo))

    item = listar_itens_emprestimo(emprestimo.id_emprestimo)[0]
    item.preco_pago = Decimal("13.00")
    print("atualizou item? ", atualizar_item_emprestimo(item))
    print("apagou item?    ", deletar_item_emprestimo(emprestimo.id_emprestimo, jogo_2.id_jogo))
    print("itens:          ", listar_itens_emprestimo(emprestimo.id_emprestimo))

    print("apagou empr.?   ", deletar_emprestimo(emprestimo.id_emprestimo))

    # aluguel com um jogo que não existe (-1): o primeiro item entra, o segundo falha,
    # e o rollback desfaz tudo — não sobra empréstimo nenhum desse cliente
    print("jogo inexistente:", registrar_aluguel(cliente.id_cliente, [jogo_1.id_jogo, -1]))
    print("sobrou algo?     ", [e for e in listar_emprestimos() if e.id_cliente == cliente.id_cliente])

if cliente:
    deletar_cliente(cliente.id_cliente)
if jogo_1:
    deletar_jogo(jogo_1.id_jogo)
if jogo_2:
    deletar_jogo(jogo_2.id_jogo)

# 10. Menu

O programa interativo da locadora. Ele **não fala com o banco diretamente**: só lê o que o usuário digita (`input`) e chama os métodos das seções 3 a 9.

| Opção | Função do menu | Métodos usados |
|---|---|---|
| 1 | `alugar()` | `listar_clientes`, `listar_jogos`, `registrar_aluguel` |
| 2 | `devolver()` | `listar_emprestimos`, `registrar_devolucao` |
| 3 | `cadastrar_cliente()` | `criar_cliente`, `criar_endereco` |
| 4 | `cadastrar_jogo()` | `criar_jogo`, `vincular_jogo_plataforma` |
| 5 | `cadastrar_plataformas()` | `criar_plataforma` |
| 6 | `ver_plataformas()` | `listar_plataformas` |
| 7 | `ver_jogos()` | `listar_jogos`, `listar_plataformas_do_jogo` |
| 8 | `ver_clientes()` | `listar_clientes`, `buscar_endereco_do_cliente` |
| 9 | `ver_emprestimos()` | `listar_emprestimos`, `listar_itens_emprestimo` |
| 0 | sair | — |

`ler_inteiro` e `ler_decimal` evitam que o programa quebre quando o usuário digita uma letra onde era esperado um número.

In [ ]:
def ler_inteiro(mensagem: str) -> int | None:
    try:
        return int(input(mensagem))
    except ValueError:  # o usuário digitou algo que não é número
        print("Valor inválido, digite um número inteiro")
        return None


def ler_decimal(mensagem: str) -> Decimal | None:
    texto = input(mensagem).replace(",", ".")  # aceita 12,50 e 12.50
    try:
        return Decimal(texto)
    except ArithmeticError:  # Decimal("abc") levanta InvalidOperation, que é um ArithmeticError
        print("Valor inválido, digite um número (ex.: 12.50)")
        return None


def ver_plataformas():
    for plataforma in listar_plataformas():
        print(plataforma)


def ver_jogos():
    for jogo in listar_jogos():
        print(jogo, "->", listar_plataformas_do_jogo(jogo.id_jogo))


def ver_clientes():
    for cliente in listar_clientes():
        print(cliente)
        endereco = buscar_endereco_do_cliente(cliente.id_cliente)
        if endereco:
            print("    ", endereco)


def ver_emprestimos():
    for emprestimo in listar_emprestimos():
        print(emprestimo)
        for item in listar_itens_emprestimo(emprestimo.id_emprestimo):
            print("    ", item)


def cadastrar_plataformas():
    while True:
        print("Deseja inserir uma plataforma (S para sim, N para não)")
        desejo = input().upper()

        if desejo == 'S':
            print("Qual o nome da plataforma?")
            nome = input()
            if nome == "":
                print("Plataforma inválida")
                continue
            print("Qual o nome do fabricante?")
            fabricante = input()
            if fabricante == "":
                print("Fabricante inválido")
                continue
            plataforma = criar_plataforma(nome, fabricante)
            if not plataforma:
                print("Não foi possível inserir uma plataforma")
                continue
            print(f"plataforma inserida com sucesso {plataforma}")
        elif desejo == 'N':
            break
        else:
            continue


def cadastrar_jogo():
    titulo = input("Título: ")
    if titulo == "":
        print("Título inválido")
        return
    genero = input("Gênero: ") or None  # vazio vira None, que vira NULL no banco
    ano = ler_inteiro("Ano de lançamento: ")
    preco = ler_decimal("Preço da diária: ")
    if ano is None or preco is None:
        return

    jogo = criar_jogo(titulo, genero, ano, preco)
    if not jogo:
        print("Não foi possível cadastrar o jogo")
        return
    print(f"jogo cadastrado com sucesso {jogo}")

    ver_plataformas()
    ids = input("Ids das plataformas em que o jogo roda, separados por vírgula: ")
    for id_texto in ids.split(","):
        if id_texto.strip().isdigit():
            vincular_jogo_plataforma(jogo.id_jogo, int(id_texto))
    print("Plataformas do jogo:", listar_plataformas_do_jogo(jogo.id_jogo))


def cadastrar_cliente():
    nome = input("Nome: ")
    if nome == "":
        print("Nome inválido")
        return
    telefone = input("Telefone: ") or None
    email = input("E-mail: ") or None

    cliente = criar_cliente(nome, telefone, email)
    if not cliente:
        print("Não foi possível cadastrar o cliente")
        return
    print(f"cliente cadastrado com sucesso {cliente}")

    if input("Cadastrar endereço? (S/N) ").upper() == 'S':
        logradouro = input("Logradouro: ")
        numero = input("Número: ") or None
        bairro = input("Bairro: ") or None
        cidade = input("Cidade: ")
        uf = input("UF: ").upper() or None
        cep = input("CEP: ") or None
        endereco = criar_endereco(cliente.id_cliente, logradouro, cidade, numero, bairro, uf, cep)
        if endereco:
            print(f"endereço cadastrado com sucesso {endereco}")
        else:
            print("Não foi possível cadastrar o endereço")


def alugar():
    print(listar_clientes(apenas_ativos=True))
    id_cliente = ler_inteiro("Id do cliente: ")
    if id_cliente is None:
        return

    print(listar_jogos())
    ids = input("Ids dos jogos, separados por vírgula: ")
    ids_jogos = [int(i) for i in ids.split(",") if i.strip().isdigit()]
    if not ids_jogos:
        print("Nenhum jogo informado")
        return

    dias = ler_inteiro("Quantos dias? ")
    if dias is None:
        return

    emprestimo = registrar_aluguel(id_cliente, ids_jogos, dias)
    if emprestimo:
        print(f"aluguel registrado com sucesso {emprestimo}")
        print("itens:", listar_itens_emprestimo(emprestimo.id_emprestimo))
    else:
        print("Não foi possível registrar o aluguel")


def devolver():
    abertos = listar_emprestimos(apenas_abertos=True)
    if not abertos:
        print("Nenhum empréstimo em aberto")
        return
    for emprestimo in abertos:
        print(emprestimo)

    id_emprestimo = ler_inteiro("Id do empréstimo: ")
    if id_emprestimo is None:
        return
    multa = ler_decimal("Multa (0 se não houver): ")
    if multa is None:
        return

    if registrar_devolucao(id_emprestimo, date.today(), multa):
        print("devolução registrada com sucesso", buscar_emprestimo(id_emprestimo))
    else:
        print("Não foi possível registrar a devolução (id errado ou já devolvido?)")

In [ ]:
while True:
    print()
    print("Bem vindo a locadora")
    print("1 - Alugar")
    print("2 - Devolver")
    print("3 - Cadastrar cliente")
    print("4 - Cadastrar jogo")
    print("5 - Cadastrar plataformas")
    print("6 - Ver plataformas")
    print("7 - Ver jogos")
    print("8 - Ver clientes")
    print("9 - Ver empréstimos")
    print("0 - Sair")
    desejo = input("O que deseja fazer? ")

    if desejo == "1":
        alugar()
    elif desejo == "2":
        devolver()
    elif desejo == "3":
        cadastrar_cliente()
    elif desejo == "4":
        cadastrar_jogo()
    elif desejo == "5":
        cadastrar_plataformas()
    elif desejo == "6":
        ver_plataformas()
    elif desejo == "7":
        ver_jogos()
    elif desejo == "8":
        ver_clientes()
    elif desejo == "9":
        ver_emprestimos()
    elif desejo == "0":
        print("Até logo!")
        break
    else:
        print("Opção inválida")

## Encerrando

Ao terminar, feche o cursor e a conexão para liberar o servidor.

In [ ]:
cursor.close()
conexao.close()
print("conexão encerrada")